# Compute Roundtrip Fidelity

CrossCell bidirectional roundtrip + mixed roundtrip with other tools.

- **RDS roundtrip**: RDS → H5AD → RDS (42 Seurat datasets, CrossCell only)
- **H5AD roundtrip**: H5AD → RDS → H5AD (13 AnnData datasets, CrossCell only)
- **Mixed roundtrip**: Other tool RDS→H5AD, then CrossCell H5AD→RDS→H5AD, compare (4 tools × 42 datasets)

Computes 7 fidelity metrics: Pearson R, Spearman ρ, MSE, RMSE, max_diff, mean_diff, exact_match_pct.

**Output**: `benchmark/results/roundtrip_fidelity.json`


In [8]:
import warnings
warnings.filterwarnings('ignore')

import json, subprocess, time, traceback
from pathlib import Path
import numpy as np
import anndata as ad
import scipy.sparse as sp
from scipy.stats import pearsonr, spearmanr

RESULTS_DIR = Path('/benchmark/results')
DATA_DIR = Path('/benchmark/data/generated')
TMP = Path('/tmp/roundtrip_work')
TMP.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = RESULTS_DIR / 'roundtrip_fidelity.json'
ROBUSTNESS_FILE = RESULTS_DIR / 'robustness_benchmark.json'

OTHER_TOOLS = ['Zellkonverter', 'anndataR', 'convert2anndata', 'easySCF']

print('Setup complete')


Setup complete


## Helper Functions


In [9]:
def run_crosscell(input_path, output_path, fmt='anndata'):
    t0 = time.time()
    r = subprocess.run(
        ['crosscell', 'convert', '-i', input_path, '-o', output_path, '-f', fmt],
        capture_output=True, text=True, timeout=600
    )
    elapsed = time.time() - t0
    return r.returncode == 0, r.stderr, elapsed

def get_X_matrix(adata):
    if adata.X is not None:
        return adata.X
    if adata.layers:
        for key in ['counts', 'data', 'X']:
            if key in adata.layers:
                return adata.layers[key]
        return adata.layers[list(adata.layers.keys())[0]]
    return None


In [10]:
def compute_metrics(X1, X2, max_n=500000):
    if X1.shape != X2.shape:
        return None, f'shape mismatch {X1.shape} vs {X2.shape}'
    total = X1.shape[0] * X1.shape[1]
    if total <= max_n:
        f1 = np.asarray(X1.todense() if sp.issparse(X1) else X1).flatten()
        f2 = np.asarray(X2.todense() if sp.issparse(X2) else X2).flatten()
    else:
        np.random.seed(42)
        ri = np.random.randint(0, X1.shape[0], max_n)
        ci = np.random.randint(0, X1.shape[1], max_n)
        if sp.issparse(X1):
            f1 = np.array([X1[r, c] for r, c in zip(ri, ci)]).flatten()
        else:
            f1 = X1[ri, ci].flatten()
        if sp.issparse(X2):
            f2 = np.array([X2[r, c] for r, c in zip(ri, ci)]).flatten()
        else:
            f2 = X2[ri, ci].flatten()
    if np.std(f1) == 0 or np.std(f2) == 0:
        pr = 1.0 if np.allclose(f1, f2) else 0.0
        sr = pr
    else:
        pr, _ = pearsonr(f1, f2)
        sr, _ = spearmanr(f1, f2)
    diff = np.abs(f1 - f2)
    mse = float(np.mean(diff**2))
    exact = np.sum(np.isclose(f1, f2, rtol=1e-7, atol=1e-10))
    return {
        'pearson_r': float(pr), 'spearman_rho': float(sr),
        'mse': mse, 'rmse': float(np.sqrt(mse)),
        'max_diff': float(np.max(diff)), 'mean_diff': float(np.mean(diff)),
        'exact_match_pct': float(exact / len(f1) * 100),
    }, None

print('Helpers defined')


Helpers defined


In [11]:
def run_r_tool(rds_path, h5ad_path, tool_name):
    scripts = {
        'Zellkonverter': (
            'suppressPackageStartupMessages({library(zellkonverter);library(Seurat);library(SingleCellExperiment)})\n'
            f'obj <- readRDS("{rds_path}"); tryCatch({{obj <- UpdateSeuratObject(obj)}}, error=function(e){{}})\n'
            f'sce <- as.SingleCellExperiment(obj)\nwriteH5AD(sce, "{h5ad_path}")'
        ),
        'anndataR': (
            'suppressPackageStartupMessages({library(anndataR);library(Seurat)})\n'
            f'obj <- readRDS("{rds_path}"); tryCatch({{obj <- UpdateSeuratObject(obj)}}, error=function(e){{}})\n'
            f'if (file.exists("{h5ad_path}")) file.remove("{h5ad_path}")\n'
            f'ad <- as_AnnData(obj)\nwrite_h5ad(ad, "{h5ad_path}")'
        ),
        'convert2anndata': (
            'suppressPackageStartupMessages({library(convert2anndata);library(Seurat)})\n'
            f'obj <- readRDS("{rds_path}"); tryCatch({{obj <- UpdateSeuratObject(obj)}}, error=function(e){{}})\n'
            f'sce <- convert_seurat_to_sce(obj)\nad <- convert_to_anndata(sce)\n'
            f'anndata::write_h5ad(ad, "{h5ad_path}")'
        ),
        'easySCF': (
            'suppressPackageStartupMessages({library(easySCFr);library(Seurat)})\n'
            f'obj <- readRDS("{rds_path}"); tryCatch({{obj <- UpdateSeuratObject(obj)}}, error=function(e){{}})\n'
            f'saveH5(obj, "{h5ad_path}")'
        ),
    }
    if tool_name not in scripts:
        return False, f'Unknown tool: {tool_name}'
    r = subprocess.run(
        ['Rscript', '-e', scripts[tool_name]],
        capture_output=True, text=True, timeout=600
    )
    return r.returncode == 0, r.stderr

def safe_read_h5ad(h5ad_path, tool_name=''):
    import h5py
    try:
        with h5py.File(h5ad_path, 'r') as f:
            keys = list(f.keys())
    except Exception as e:
        return None, f'not valid HDF5: {e}'
    if 'X' not in keys and 'assay' in keys:
        try:
            from easySCFpy import loadH5
            adata = loadH5(h5ad_path)
            try:
                if adata.raw is not None:
                    adata = ad.AnnData(X=adata.raw.X)
            except Exception:
                pass
            return adata, None
        except Exception as e:
            return None, f'easySCF loadH5 error: {e}'
    if 'X' not in keys and 'obs' not in keys:
        return None, f'non-standard H5 format, keys={keys}'
    try:
        adata = ad.read_h5ad(h5ad_path)
        return adata, None
    except Exception as e:
        return None, str(e)

def extract_error_reason(stderr_text):
    if not stderr_text:
        return 'unknown error'
    for line in stderr_text.split('\n'):
        line = line.strip()
        if line.startswith('Error') or 'error' in line.lower():
            return line[:120]
    lines = [l.strip() for l in stderr_text.strip().split('\n') if l.strip()]
    return lines[-1][:120] if lines else 'unknown error'

print('R tool helpers defined')


R tool helpers defined


## Load Dataset List


In [12]:
with open(ROBUSTNESS_FILE) as f:
    robustness = json.load(f)

# RDS datasets (42)
RDS_DATASETS = []
for entry in robustness['crosscell']['rds_to_h5ad']:
    RDS_DATASETS.append({
        'test_id': entry['test_id'],
        'file': entry['file'],
        'dataset': entry['dataset'],
        'n_cells': entry.get('n_cells', 0),
        'n_genes': entry.get('n_genes', 0),
    })

# H5AD datasets (13)
H5AD_DATASETS = []
for entry in robustness['crosscell']['h5ad_to_rds']:
    H5AD_DATASETS.append({
        'test_id': entry['test_id'],
        'file': entry['file'],
        'dataset': entry.get('dataset', entry['test_id']),
        'n_cells': entry.get('n_cells', 0),
        'n_genes': entry.get('n_genes', 0),
    })

print(f'RDS datasets: {len(RDS_DATASETS)}')
print(f'H5AD datasets: {len(H5AD_DATASETS)}')


RDS datasets: 42
H5AD datasets: 13


## RDS Roundtrip: RDS → H5AD → RDS

For each Seurat RDS file, convert to H5AD then back to RDS, then compare expression matrices.


In [13]:
rds_roundtrip_results = []
print('=== RDS Roundtrip (RDS -> H5AD -> RDS) ===')
for i, ds in enumerate(RDS_DATASETS):
    tid = ds['test_id']
    rds_path = str(DATA_DIR / ds['file'])
    if not Path(rds_path).exists():
        print(f'  [{i+1}/{len(RDS_DATASETS)}] {tid}: NOT FOUND, skip')
        continue

    h5ad_tmp = str(TMP / f'{tid}_step1.h5ad')
    rds_rt = str(TMP / f'{tid}_step2.rds')

    print(f'  [{i+1}/{len(RDS_DATASETS)}] {tid}...', end=' ')

    # Step 1: RDS -> H5AD
    ok1, err1, t1 = run_crosscell(rds_path, h5ad_tmp, 'anndata')
    if not ok1:
        print(f'RDS->H5AD FAILED: {err1[:100]}')
        rds_roundtrip_results.append({
            'dataset': tid, 'direction': 'rds_h5ad_rds', 'status': 'failed',
            'error_message': err1[:200], **{k: None for k in ['pearson_r','spearman_rho','max_diff','mean_diff','rmse','mse','exact_match_pct']},
            'n_cells': ds['n_cells'], 'n_genes': ds['n_genes'], 'conversion_time_s': t1,
        })
        continue

    # Step 2: H5AD -> RDS
    ok2, err2, t2 = run_crosscell(h5ad_tmp, rds_rt, 'seurat')
    if not ok2:
        print(f'H5AD->RDS FAILED: {err2[:100]}')
        rds_roundtrip_results.append({
            'dataset': tid, 'direction': 'rds_h5ad_rds', 'status': 'failed',
            'error_message': err2[:200], **{k: None for k in ['pearson_r','spearman_rho','max_diff','mean_diff','rmse','mse','exact_match_pct']},
            'n_cells': ds['n_cells'], 'n_genes': ds['n_genes'], 'conversion_time_s': t1 + t2,
        })
        continue

    # Step 3: Read original RDS via CrossCell (convert to H5AD for comparison)
    # We compare the original H5AD (step1) with a re-converted H5AD from the roundtrip RDS
    h5ad_rt = str(TMP / f'{tid}_step3.h5ad')
    ok3, err3, t3 = run_crosscell(rds_rt, h5ad_rt, 'anndata')
    if not ok3:
        print(f'RDS(rt)->H5AD FAILED')
        continue

    try:
        adata_orig = ad.read_h5ad(h5ad_tmp)
        adata_rt = ad.read_h5ad(h5ad_rt)
        X_orig = get_X_matrix(adata_orig)
        X_rt = get_X_matrix(adata_rt)
        if X_orig is None or X_rt is None:
            print('X is None')
            continue
        metrics, err = compute_metrics(X_orig, X_rt)
        if metrics:
            total_time = t1 + t2 + t3
            print(f"R={metrics['pearson_r']:.10f} MSE={metrics['mse']:.2e} Match={metrics['exact_match_pct']:.1f}% ({total_time:.1f}s)")
            rds_roundtrip_results.append({
                'dataset': tid, 'direction': 'rds_h5ad_rds', 'status': 'success',
                **metrics, 'ari': None, 'nmi': None,
                'n_cells': ds['n_cells'], 'n_genes': ds['n_genes'],
                'conversion_time_s': total_time, 'error_message': None,
            })
        else:
            print(f'metrics error: {err}')
    except Exception as e:
        print(f'ERROR: {e}')

print(f'\nRDS roundtrip: {len([r for r in rds_roundtrip_results if r["status"]=="success"])}/{len(RDS_DATASETS)} success')


=== RDS Roundtrip (RDS -> H5AD -> RDS) ===
  [1/42] v4_pbmc3k_raw... R=1.0000000000 MSE=0.00e+00 Match=100.0% (3.3s)
  [2/42] v4_pbmc3k_processed... R=1.0000000000 MSE=0.00e+00 Match=100.0% (3.7s)
  [3/42] v5_pbmc3k_raw... R=1.0000000000 MSE=0.00e+00 Match=100.0% (3.1s)
  [4/42] v5_pbmc3k_processed... R=1.0000000000 MSE=0.00e+00 Match=100.0% (4.9s)
  [5/42] v4_stxKidney_raw... R=1.0000000000 MSE=0.00e+00 Match=100.0% (12.6s)
  [6/42] v5_stxKidney_raw... R=1.0000000000 MSE=0.00e+00 Match=100.0% (12.1s)
  [7/42] v4_celegans.embryo_raw... R=1.0000000000 MSE=0.00e+00 Match=100.0% (8.2s)
  [8/42] v4_celegans.embryo_processed... R=1.0000000000 MSE=0.00e+00 Match=100.0% (8.4s)
  [9/42] v5_celegans.embryo_raw... R=1.0000000000 MSE=0.00e+00 Match=100.0% (9.2s)
  [10/42] v5_celegans.embryo_processed... R=1.0000000000 MSE=0.00e+00 Match=100.0% (8.2s)
  [11/42] v4_cbmc_raw... R=1.0000000000 MSE=0.00e+00 Match=100.0% (13.2s)
  [12/42] v4_cbmc_processed... R=1.0000000000 MSE=0.00e+00 Match=100.0% (1

## H5AD Roundtrip: H5AD → RDS → H5AD

For each AnnData H5AD file, convert to RDS then back to H5AD, then compare expression matrices.


In [14]:
h5ad_roundtrip_results = []
print('=== H5AD Roundtrip (H5AD -> RDS -> H5AD) ===')
for i, ds in enumerate(H5AD_DATASETS):
    tid = ds['test_id']
    h5ad_path = str(DATA_DIR / ds['file'])
    if not Path(h5ad_path).exists():
        print(f'  [{i+1}/{len(H5AD_DATASETS)}] {tid}: NOT FOUND, skip')
        continue

    rds_tmp = str(TMP / f'{tid}_h2r.rds')
    h5ad_rt = str(TMP / f'{tid}_h2r_rt.h5ad')

    print(f'  [{i+1}/{len(H5AD_DATASETS)}] {tid}...', end=' ')

    # Step 1: H5AD -> RDS
    ok1, err1, t1 = run_crosscell(h5ad_path, rds_tmp, 'seurat')
    if not ok1:
        print(f'H5AD->RDS FAILED: {err1[:100]}')
        h5ad_roundtrip_results.append({
            'dataset': tid, 'direction': 'h5ad_rds_h5ad', 'status': 'failed',
            'error_message': err1[:200], **{k: None for k in ['pearson_r','spearman_rho','max_diff','mean_diff','rmse','mse','exact_match_pct']},
            'n_cells': ds['n_cells'], 'n_genes': ds['n_genes'], 'conversion_time_s': t1,
        })
        continue

    # Step 2: RDS -> H5AD
    ok2, err2, t2 = run_crosscell(rds_tmp, h5ad_rt, 'anndata')
    if not ok2:
        print(f'RDS->H5AD FAILED: {err2[:100]}')
        h5ad_roundtrip_results.append({
            'dataset': tid, 'direction': 'h5ad_rds_h5ad', 'status': 'failed',
            'error_message': err2[:200], **{k: None for k in ['pearson_r','spearman_rho','max_diff','mean_diff','rmse','mse','exact_match_pct']},
            'n_cells': ds['n_cells'], 'n_genes': ds['n_genes'], 'conversion_time_s': t1 + t2,
        })
        continue

    try:
        adata_orig = ad.read_h5ad(h5ad_path)
        adata_rt = ad.read_h5ad(h5ad_rt)
        X_orig = get_X_matrix(adata_orig)
        X_rt = get_X_matrix(adata_rt)
        if X_orig is None or X_rt is None:
            print('X is None')
            continue
        metrics, err = compute_metrics(X_orig, X_rt)
        if metrics:
            total_time = t1 + t2
            print(f"R={metrics['pearson_r']:.10f} MSE={metrics['mse']:.2e} Match={metrics['exact_match_pct']:.1f}% ({total_time:.1f}s)")
            h5ad_roundtrip_results.append({
                'dataset': tid, 'direction': 'h5ad_rds_h5ad', 'status': 'success',
                **metrics, 'ari': None, 'nmi': None,
                'n_cells': ds['n_cells'], 'n_genes': ds['n_genes'],
                'conversion_time_s': total_time, 'error_message': None,
            })
        else:
            print(f'metrics error: {err}')
    except Exception as e:
        print(f'ERROR: {e}')

print(f'\nH5AD roundtrip: {len([r for r in h5ad_roundtrip_results if r["status"]=="success"])}/{len(H5AD_DATASETS)} success')


=== H5AD Roundtrip (H5AD -> RDS -> H5AD) ===
  [1/13] scanpy_pbmc3k... R=1.0000000000 MSE=0.00e+00 Match=100.0% (2.8s)
  [2/13] scanpy_pbmc3k_processed... R=1.0000000000 MSE=0.00e+00 Match=100.0% (9.7s)
  [3/13] scvelo_dentategyrus... R=1.0000000000 MSE=0.00e+00 Match=100.0% (14.3s)
  [4/13] scvelo_pancreas... R=1.0000000000 MSE=0.00e+00 Match=100.0% (29.6s)
  [5/13] cellxgene_pbmc_15k... R=1.0000000000 MSE=0.00e+00 Match=100.0% (23.4s)
  [6/13] cellxgene_heart_23k... R=1.0000000000 MSE=0.00e+00 Match=100.0% (53.7s)
  [7/13] cellxgene_brain_40k... R=1.0000000000 MSE=0.00e+00 Match=100.0% (343.8s)
  [8/13] squidpy_visium_hne... R=1.0000000000 MSE=0.00e+00 Match=100.0% (23.9s)
  [9/13] squidpy_mibitof... R=1.0000000000 MSE=0.00e+00 Match=100.0% (0.7s)
  [10/13] squidpy_slideseqv2... R=1.0000000000 MSE=0.00e+00 Match=100.0% (19.9s)
  [11/13] squidpy_seqfish... R=1.0000000000 MSE=0.00e+00 Match=100.0% (2.9s)
  [12/13] squidpy_merfish... R=1.0000000000 MSE=0.00e+00 Match=100.0% (8.2s)
  [13

## Mixed Roundtrip: Other Tool RDS→H5AD, then CrossCell H5AD→RDS→H5AD

For each other tool that successfully converts RDS→H5AD, we use CrossCell to do
H5AD→RDS→H5AD roundtrip, then compare the tool's original H5AD with the roundtrip result.

This measures how much information each tool preserves through a full roundtrip cycle,
using CrossCell as the roundtrip engine.


In [15]:
# Load robustness data to know which tool/dataset combos succeeded
with open(ROBUSTNESS_FILE) as f:
    robustness = json.load(f)

success_map = {}
for tool_key in ['zellkonverter', 'anndataR', 'convert2anndata', 'easySCF']:
    tool_name = {'zellkonverter': 'Zellkonverter'}.get(tool_key, tool_key)
    success_map[tool_name] = {}
    for entry in robustness.get(tool_key, {}).get('rds_to_h5ad', []):
        success_map[tool_name][entry['test_id']] = (entry['status'] == 'success')

mixed_roundtrip_results = []

for tool in OTHER_TOOLS:
    print(f'\n=== {tool} Mixed Roundtrip ===')
    tool_count = 0
    tool_success = 0

    for i, ds in enumerate(RDS_DATASETS):
        tid = ds['test_id']
        rds_path = str(DATA_DIR / ds['file'])

        if not Path(rds_path).exists():
            continue
        if not success_map.get(tool, {}).get(tid, False):
            continue

        tool_h5ad = str(TMP / f'mixed_{tool}_{tid}.h5ad')
        rt_rds = str(TMP / f'mixed_{tool}_{tid}_rt.rds')
        rt_h5ad = str(TMP / f'mixed_{tool}_{tid}_rt.h5ad')

        tool_count += 1
        print(f'  [{tool_count}] {tid}...', end=' ')

        # Step 1: Other tool RDS -> H5AD
        try:
            ok1, stderr1 = run_r_tool(rds_path, tool_h5ad, tool)
        except subprocess.TimeoutExpired:
            print('TIMEOUT (step1)')
            mixed_roundtrip_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'mixed_roundtrip',
                'dataset': ds['dataset'], 'status': 'timeout',
            })
            continue
        except Exception as e:
            print(f'ERROR (step1): {e}')
            continue

        if not ok1:
            print('FAILED (step1)')
            mixed_roundtrip_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'mixed_roundtrip',
                'dataset': ds['dataset'], 'status': 'convert_failed',
                'error': extract_error_reason(stderr1),
            })
            continue

        # Step 2: CrossCell H5AD -> RDS
        ok2, stderr2, t2 = run_crosscell(tool_h5ad, rt_rds, 'seurat')
        if not ok2:
            print('FAILED (CC h5ad->rds)')
            mixed_roundtrip_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'mixed_roundtrip',
                'dataset': ds['dataset'], 'status': 'cc_h2r_failed',
                'error': stderr2[:200],
            })
            continue

        # Step 3: CrossCell RDS -> H5AD
        ok3, stderr3, t3 = run_crosscell(rt_rds, rt_h5ad, 'anndata')
        if not ok3:
            print('FAILED (CC rds->h5ad)')
            mixed_roundtrip_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'mixed_roundtrip',
                'dataset': ds['dataset'], 'status': 'cc_r2h_failed',
                'error': stderr3[:200],
            })
            continue

        # Step 4: Compare tool's H5AD vs roundtrip H5AD
        try:
            adata_orig, err_orig = safe_read_h5ad(tool_h5ad, tool)
            if err_orig:
                print(f'read error (orig): {err_orig}')
                mixed_roundtrip_results.append({
                    'test_id': tid, 'tool': tool, 'test_type': 'mixed_roundtrip',
                    'dataset': ds['dataset'], 'status': 'read_error',
                    'error': err_orig[:200],
                })
                continue

            adata_rt = ad.read_h5ad(rt_h5ad)
            X_orig = get_X_matrix(adata_orig)
            X_rt = get_X_matrix(adata_rt)

            if X_orig is None or X_rt is None:
                print('X is None')
                continue

            metrics, err = compute_metrics(X_orig, X_rt)
            if metrics:
                tool_success += 1
                print(f"R={metrics['pearson_r']:.10f} MSE={metrics['mse']:.2e} Match={metrics['exact_match_pct']:.1f}%")
                mixed_roundtrip_results.append({
                    'test_id': tid, 'tool': tool, 'test_type': 'mixed_roundtrip',
                    'dataset': ds['dataset'], 'status': 'success',
                    'n_cells': ds['n_cells'], 'n_genes': ds['n_genes'],
                    **metrics,
                })
            else:
                print(f'metrics error: {err}')
        except Exception as e:
            print(f'ERROR: {e}')
            mixed_roundtrip_results.append({
                'test_id': tid, 'tool': tool, 'test_type': 'mixed_roundtrip',
                'dataset': ds['dataset'], 'status': 'error',
                'error': str(e)[:200],
            })

    print(f'{tool}: {tool_success}/{tool_count} success')

print(f'\nTotal mixed roundtrip results: {len(mixed_roundtrip_results)}')



=== Zellkonverter Mixed Roundtrip ===
  [1] v4_pbmc3k_raw... R=1.0000000000 MSE=0.00e+00 Match=100.0%
  [2] v4_pbmc3k_processed... R=1.0000000000 MSE=0.00e+00 Match=100.0%
  [3] v5_pbmc3k_raw... R=1.0000000000 MSE=0.00e+00 Match=100.0%
  [4] v5_pbmc3k_processed... R=1.0000000000 MSE=0.00e+00 Match=100.0%
  [5] v4_stxKidney_raw... R=1.0000000000 MSE=0.00e+00 Match=100.0%
  [6] v5_stxKidney_raw... R=1.0000000000 MSE=0.00e+00 Match=100.0%
  [7] v4_celegans.embryo_raw... R=1.0000000000 MSE=0.00e+00 Match=100.0%
  [8] v4_celegans.embryo_processed... R=1.0000000000 MSE=0.00e+00 Match=100.0%
  [9] v5_celegans.embryo_raw... R=1.0000000000 MSE=0.00e+00 Match=100.0%
  [10] v5_celegans.embryo_processed... R=1.0000000000 MSE=0.00e+00 Match=100.0%
  [11] v4_cbmc_raw... R=1.0000000000 MSE=0.00e+00 Match=100.0%
  [12] v4_cbmc_processed... R=1.0000000000 MSE=0.00e+00 Match=100.0%
  [13] v5_cbmc_raw... R=1.0000000000 MSE=0.00e+00 Match=100.0%
  [14] v5_cbmc_processed... R=1.0000000000 MSE=0.00e+00 Mat

## Save Results


In [16]:
rds_success = [r for r in rds_roundtrip_results if r['status'] == 'success']
h5ad_success = [r for r in h5ad_roundtrip_results if r['status'] == 'success']

results = {
    'rds_roundtrip': rds_roundtrip_results,
    'h5ad_roundtrip': h5ad_roundtrip_results,
    'mixed_roundtrip': mixed_roundtrip_results,
    'cross_tool_compatibility': {
        'h5ad_readable': True,
        'rds_readable': True,
    },
    'summary': {
        'rds_roundtrip_total': len(RDS_DATASETS),
        'rds_roundtrip_success': len(rds_success),
        'h5ad_roundtrip_total': len(H5AD_DATASETS),
        'h5ad_roundtrip_success': len(h5ad_success),
        'avg_pearson_r': float(np.mean([r['pearson_r'] for r in rds_success + h5ad_success])) if rds_success + h5ad_success else None,
        'avg_spearman_rho': float(np.mean([r['spearman_rho'] for r in rds_success + h5ad_success])) if rds_success + h5ad_success else None,
        'avg_exact_match_pct': float(np.mean([r['exact_match_pct'] for r in rds_success + h5ad_success])) if rds_success + h5ad_success else None,
        'avg_rmse': float(np.mean([r['rmse'] for r in rds_success + h5ad_success])) if rds_success + h5ad_success else None,
    }
}

with open(OUTPUT_FILE, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved to {OUTPUT_FILE}')

print(f'\n=== Summary ===')
print(f'CrossCell RDS roundtrip: {len(rds_success)}/{len(RDS_DATASETS)}')
print(f'CrossCell H5AD roundtrip: {len(h5ad_success)}/{len(H5AD_DATASETS)}')
if rds_success + h5ad_success:
    all_s = rds_success + h5ad_success
    print(f'  Avg Pearson R: {np.mean([r["pearson_r"] for r in all_s]):.10f}')
    print(f'  Avg MSE: {np.mean([r["mse"] for r in all_s]):.2e}')
    print(f'  Avg Exact Match: {np.mean([r["exact_match_pct"] for r in all_s]):.1f}%')

print(f'\nMixed roundtrip:')
for tool in OTHER_TOOLS:
    tool_res = [r for r in mixed_roundtrip_results if r['tool'] == tool and r.get('status') == 'success']
    tool_fail = [r for r in mixed_roundtrip_results if r['tool'] == tool and r.get('status') != 'success']
    print(f'  {tool}: {len(tool_res)} success, {len(tool_fail)} failed')
    if tool_res:
        avg_r = np.mean([r['pearson_r'] for r in tool_res])
        avg_mse = np.mean([r['mse'] for r in tool_res])
        avg_em = np.mean([r['exact_match_pct'] for r in tool_res])
        print(f'    Avg Pearson R: {avg_r:.10f}, MSE: {avg_mse:.2e}, Match: {avg_em:.1f}%')


Saved to /benchmark/results/roundtrip_fidelity.json

=== Summary ===
CrossCell RDS roundtrip: 42/42
CrossCell H5AD roundtrip: 13/13
  Avg Pearson R: 1.0000000000
  Avg MSE: 0.00e+00
  Avg Exact Match: 100.0%

Mixed roundtrip:
  Zellkonverter: 42 success, 0 failed
    Avg Pearson R: 1.0000000000, MSE: 0.00e+00, Match: 100.0%
  anndataR: 0 success, 42 failed
  convert2anndata: 17 success, 2 failed
    Avg Pearson R: 1.0000000000, MSE: 0.00e+00, Match: 100.0%
  easySCF: 0 success, 42 failed
